# TabNet: NFL Draft 2-Year AV Prediction

Implements the TabNet architecture from *Arik & Pfister, AAAI 2021* (`arXiv 1908.07442`) for predicting 2-year Approximate Value (AV) from draft-night features.

**Key TabNet properties used here:**
- Sequential attention selects a sparse subset of features at each decision step
- Prior scale `P[i]` penalises re-selecting features already used
- Sparsity entropy loss encourages focused feature selection
- Aggregated masks provide model-native feature importance

**Walk-forward structure (mirrors model_v1):**  
Train on all years < test_year, evaluate on test_year, step forward.

## 0. Imports & configuration

In [ ]:
import os, sys, glob
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ExponentialLR
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Make sure repo root is on path so `src` is importable
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.model_v2.tabnet import TabNetRegressor

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
DRAFT_CSV = os.path.join(REPO_ROOT, 'src', 'data', 'raw', 'draft_picks.csv')
AV_DIR    = os.path.join(REPO_ROOT, 'scraping_av', 'data')
OUT_DIR   = os.path.join(REPO_ROOT, 'poc_outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# ── TabNet hyperparameters ────────────────────────────────────────────────────
# Tuned for our dataset size (~500-2000 training samples per fold).
# Reference values from paper appendix are for much larger datasets.
TABNET_CFG = dict(
    n_d=16,           # decision embedding width N_d
    n_a=16,           # attention embedding width N_a
    n_steps=4,        # number of sequential decision steps N_steps
    gamma=1.5,        # relaxation factor γ for prior scale
    n_shared=2,       # shared FC-BN-GLU layers in feature transformer
    n_step_dep=2,     # step-dependent layers per step
    vbs=64,           # virtual batch size for Ghost BN
    momentum=0.02,    # BN momentum
    lambda_sparse=1e-3,  # sparsity regularisation coefficient
)

# ── Training hyperparameters ──────────────────────────────────────────────────
TRAIN_CFG = dict(
    lr=0.02,
    lr_decay=0.95,
    lr_decay_steps=200,
    batch_size=256,
    max_epochs=500,
    patience=50,
    val_fraction=0.2,
)

# Feature columns to use
NUM_COLS = ['pick', 'round', 'age']
CAT_COLS = ['position', 'category', 'team', 'college', 'side']

## 1. Data loading

In [ ]:
def load_draft(path):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if 'draft_season' in df.columns and 'season' not in df.columns:
        df = df.rename(columns={'draft_season': 'season'})
    needed = ['season','pick','round','team','position','category',
              'side','age','college','pfr_player_id','pfr_player_name']
    keep = [c for c in needed if c in df.columns]
    df = df[keep].copy()
    df['season'] = df['season'].astype(int)
    df['pick']   = pd.to_numeric(df['pick'], errors='coerce')
    df = df.dropna(subset=['pick']).copy()
    df['pick'] = df['pick'].astype(int)
    if 'round' in df.columns: df['round'] = pd.to_numeric(df['round'], errors='coerce')
    if 'age'   in df.columns: df['age']   = pd.to_numeric(df['age'],   errors='coerce')
    for c in ['team','position','category','side','college']:
        if c in df.columns: df[c] = df[c].astype(str)
    return df

def load_av(av_dir):
    paths = sorted(glob.glob(os.path.join(av_dir, '*_av.csv')))
    dfs = []
    for fp in paths:
        df = pd.read_csv(fp)
        df.columns = [c.strip() for c in df.columns]
        df = df.rename(columns={'Year':'season','PlayerID':'pfr_player_id',
                                 'Player':'player_name','AV':'av'})
        df = df[['season','pfr_player_id','player_name','av']].copy()
        df['season'] = pd.to_numeric(df['season'], errors='coerce').astype('Int64')
        df['av']     = pd.to_numeric(df['av'],     errors='coerce').fillna(0.0)
        dfs.append(df)
    av = pd.concat(dfs, ignore_index=True)
    return av.groupby(['season','pfr_player_id'], as_index=False).agg(
        player_name=('player_name','first'), av=('av','sum'))

def build_labels(av_long):
    a = av_long.rename(columns={'season':'draft_season','av':'av_y'})
    b = av_long.rename(columns={'season':'next_season', 'av':'av_y1'})[['next_season','pfr_player_id','av_y1']]
    m = a.merge(b, on='pfr_player_id', how='left')
    m = m[m['next_season'] == m['draft_season'] + 1].copy()
    m['av_2yr'] = m['av_y'] + m['av_y1'].fillna(0.0)
    return m[['draft_season','pfr_player_id','av_2yr']]

draft = load_draft(DRAFT_CSV)
av_long = load_av(AV_DIR)
labels = build_labels(av_long)

df = draft.rename(columns={'season':'draft_season'}).merge(
    labels, on=['draft_season','pfr_player_id'], how='left')
df = df.dropna(subset=['av_2yr']).copy()
df['av_2yr'] = df['av_2yr'].astype(float)

print(f'Dataset: {len(df)} rows  |  years: {df.draft_season.min()} – {df.draft_season.max()}')
print(f'Target (av_2yr): mean={df.av_2yr.mean():.2f}  std={df.av_2yr.std():.2f}  max={df.av_2yr.max():.0f}')
df.head()

## 2. Preprocessing helpers

In [ ]:
def build_preprocessor(df_train):
    num_cols = [c for c in NUM_COLS if c in df_train.columns]
    cat_cols = [c for c in CAT_COLS if c in df_train.columns]
    transformers = []
    if num_cols:
        transformers.append(('num',
            Pipeline([('impute', SimpleImputer(strategy='median'))]), num_cols))
    if cat_cols:
        transformers.append(('cat',
            OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols))
    pre = ColumnTransformer(transformers, remainder='drop')
    return pre, num_cols, cat_cols

def get_feature_names(pre, num_cols, cat_cols):
    names = list(num_cols)
    if cat_cols:
        names += list(pre.named_transformers_['cat'].get_feature_names_out(cat_cols))
    return names

def to_tensor(arr):
    return torch.tensor(arr.astype(np.float32))

## 3. Training loop

In [ ]:
def train_one_fold(X_tr, y_tr, n_features, verbose=True):
    """Train TabNet with early stopping. Returns model, train_losses, val_losses."""
    model = TabNetRegressor(n_features=n_features, **TABNET_CFG).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=TRAIN_CFG['lr'])
    scheduler = ExponentialLR(optimizer, gamma=TRAIN_CFG['lr_decay'])

    n_val = max(1, int(len(X_tr) * TRAIN_CFG['val_fraction']))
    Xt, Xv = X_tr[:-n_val], X_tr[-n_val:]
    yt, yv = y_tr[:-n_val], y_tr[-n_val:]

    loader = DataLoader(TensorDataset(to_tensor(Xt), to_tensor(yt)),
                        batch_size=TRAIN_CFG['batch_size'], shuffle=True)
    Xv_t, yv_t = to_tensor(Xv).to(DEVICE), to_tensor(yv).to(DEVICE)

    best_val, best_state, patience_ctr = float('inf'), None, 0
    train_hist, val_hist = [], []
    step = 0

    for epoch in range(TRAIN_CFG['max_epochs']):
        model.train()
        ep_loss = 0.0
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred, sp, _ = model(Xb)
            loss = model.loss(pred, yb, sp)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()
            step += 1
            if step % TRAIN_CFG['lr_decay_steps'] == 0:
                scheduler.step()

        train_hist.append(ep_loss / max(1, len(loader)))

        model.eval()
        with torch.no_grad():
            vp, vsp, _ = model(Xv_t)
            vl = model.loss(vp, yv_t, vsp).item()
        val_hist.append(vl)

        if vl < best_val:
            best_val = vl
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if verbose and (epoch+1) % 100 == 0:
            print(f'  epoch {epoch+1:4d}  train={train_hist[-1]:.3f}  val={vl:.3f}  '
                  f'best={best_val:.3f}  patience={patience_ctr}/{TRAIN_CFG["patience"]}')

        if patience_ctr >= TRAIN_CFG['patience']:
            if verbose: print(f'  Early stop at epoch {epoch+1}')
            break

    model.load_state_dict(best_state)
    return model, train_hist, val_hist

## 4. Walk-forward evaluation

In [ ]:
feat_cols = [c for c in NUM_COLS + CAT_COLS if c in df.columns]
X_raw = df[feat_cols].copy()
y     = df['av_2yr'].values
years = df['draft_season'].values
unique_years = sorted(np.unique(years))

results = []
all_train_hist, all_val_hist = [], []
last_model, last_importance, last_feat_names = None, None, None
last_y_te, last_pred = None, None
latest_year = max(unique_years)

for test_year in unique_years[1:]:
    tr_mask = years < test_year
    te_mask = years == test_year

    pre, _num, _cat = build_preprocessor(X_raw[tr_mask])
    X_tr = pre.fit_transform(X_raw[tr_mask]).astype(np.float32)
    X_te = pre.transform(X_raw[te_mask]).astype(np.float32)
    feat_names = get_feature_names(pre, _num, _cat)
    y_tr, y_te = y[tr_mask], y[te_mask]

    print(f'\n=== Test {test_year}  train={tr_mask.sum()}  test={te_mask.sum()}  features={X_tr.shape[1]} ===')

    model, th, vh = train_one_fold(X_tr, y_tr, X_tr.shape[1], verbose=True)
    all_train_hist.append(th)
    all_val_hist.append(vh)

    model.eval()
    with torch.no_grad():
        Xte_t = to_tensor(X_te).to(DEVICE)
        pred_t, _, masks = model(Xte_t)
        pred = pred_t.cpu().numpy()
        importance = model.encoder.aggregate_importance(masks).cpu().numpy()

    mae  = mean_absolute_error(y_te, pred)
    rmse = float(np.sqrt(mean_squared_error(y_te, pred)))
    sp   = spearmanr(y_te, pred).correlation
    print(f'  MAE={mae:.3f}  RMSE={rmse:.3f}  Spearman={sp:.3f}')

    results.append({'test_year': int(test_year), 'n_test': int(te_mask.sum()),
                    'mae': mae, 'rmse': rmse, 'spearman': sp})

    if test_year == latest_year:
        last_model, last_importance, last_feat_names = model, importance, feat_names
        last_y_te, last_pred = y_te, pred
        last_train_hist, last_val_hist = th, vh

res_df = pd.DataFrame(results)
print('\n=== Walk-forward summary ===')
print(res_df.to_string(index=False))
res_df.to_csv(os.path.join(OUT_DIR, 'tabnet_walkforward_results.csv'), index=False)

## 5. Loss curves (final fold)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
ax.plot(last_train_hist, label='Train loss', lw=1.5)
ax.plot(last_val_hist,   label='Val loss',   lw=1.5)
ax.set_xlabel('Epoch');  ax.set_ylabel('Loss (MSE + λ·sparsity)')
ax.set_title(f'TabNet Training Curves — Test Year {latest_year}')
ax.legend()
ax.grid(True, ls='--', lw=0.5, alpha=0.4); ax.set_axisbelow(True)
for s in ['top','right']: ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f'plot_tabnet_loss_curves_{latest_year}.png'), dpi=180)
plt.show()

## 6. Predicted vs. actual (final fold)

In [ ]:
mae  = mean_absolute_error(last_y_te, last_pred)
rmse = float(np.sqrt(mean_squared_error(last_y_te, last_pred)))
sp   = spearmanr(last_y_te, last_pred).correlation

fig, ax = plt.subplots(figsize=(6.5, 5), dpi=150)
ax.scatter(last_y_te, last_pred, s=22, alpha=0.65, edgecolor='none')
lo = min(last_y_te.min(), last_pred.min())
hi = max(last_y_te.max(), last_pred.max())
ax.plot([lo, hi], [lo, hi], lw=1.2, alpha=0.8)
ax.set_title(f'TabNet: {latest_year} Draft Class', fontsize=13)
ax.set_xlabel(f'Actual 2-Year AV (AV{latest_year} + AV{latest_year+1})')
ax.set_ylabel('Predicted 2-Year AV')
ax.text(0.02, 0.98, f'MAE={mae:.2f}  RMSE={rmse:.2f}  Spearman={sp:.2f}',
        transform=ax.transAxes, va='top', ha='left', fontsize=10,
        bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.9, ec='0.85'))
ax.grid(True, ls='--', lw=0.5, alpha=0.4); ax.set_axisbelow(True)
for s in ['top','right']: ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f'plot_pred_vs_actual_tabnet_{latest_year}.png'), dpi=180)
plt.show()

## 7. Feature importance (aggregate attention masks)

In [ ]:
# Sort by importance; display top 25
top_n = 25
idx = np.argsort(last_importance)[::-1][:top_n]

fig, ax = plt.subplots(figsize=(9, 6), dpi=150)
ax.barh(
    [last_feat_names[i] for i in reversed(idx)],
    last_importance[list(reversed(idx))],
    color='#4C72B0',
)
ax.set_xlabel('Aggregate TabNet Feature Importance')
ax.set_title(f'TabNet Feature Importance — trained through {latest_year-1}')
ax.grid(True, axis='x', ls='--', lw=0.5, alpha=0.4); ax.set_axisbelow(True)
for s in ['top','right']: ax.spines[s].set_visible(False)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'plot_feature_importance_tabnet.png'), dpi=180)
plt.show()

# Print top-10 numerically
print('Top-10 features:')
for rank, i in enumerate(idx[:10], 1):
    print(f'  {rank:2d}. {last_feat_names[i]:<35s} {last_importance[i]:.4f}')

## 8. Walk-forward performance over time

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), dpi=150)
metrics = [('mae', 'MAE'), ('rmse', 'RMSE'), ('spearman', 'Spearman ρ')]

for ax, (col, label) in zip(axes, metrics):
    ax.plot(res_df['test_year'], res_df[col], marker='o', lw=1.8, ms=5)
    ax.set_title(label); ax.set_xlabel('Test Year')
    ax.grid(True, ls='--', lw=0.5, alpha=0.4); ax.set_axisbelow(True)
    for s in ['top','right']: ax.spines[s].set_visible(False)

fig.suptitle('TabNet Walk-Forward Backtesting', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'plot_tabnet_walkforward_metrics.png'), dpi=180)
plt.show()

## 9. Attention mask visualisation (step-by-step)

In [ ]:
# Show per-step attention masks for the first 30 test samples
last_model.eval()
pre_final, _num, _cat = build_preprocessor(X_raw[years < latest_year])
Xte_all = pre_final.fit_transform(X_raw[years == latest_year]).astype(np.float32)
feat_names_final = get_feature_names(pre_final, _num, _cat)

n_show = min(30, len(Xte_all))
with torch.no_grad():
    _, _, step_masks = last_model(to_tensor(Xte_all[:n_show]).to(DEVICE))

n_steps = len(step_masks)
# Only show the numeric + a few top OHE features for readability
top_feat_idx = np.argsort(last_importance)[::-1][:20]
top_feat_names = [feat_names_final[i] for i in top_feat_idx]

fig, axes = plt.subplots(1, n_steps, figsize=(4 * n_steps, 5), dpi=120)
if n_steps == 1: axes = [axes]
for s, ax in enumerate(axes):
    M_s = step_masks[s].cpu().numpy()[:n_show, :][:, top_feat_idx]  # (n_show, 20)
    im = ax.imshow(M_s.T, aspect='auto', cmap='Blues', vmin=0, vmax=M_s.max())
    ax.set_title(f'Step {s+1}', fontsize=11)
    ax.set_xlabel('Sample index')
    ax.set_yticks(range(len(top_feat_names)))
    ax.set_yticklabels(top_feat_names, fontsize=7)

fig.suptitle(f'Attention masks M[step] — {latest_year} test class (top-20 features)', fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f'plot_tabnet_masks_{latest_year}.png'), dpi=150)
plt.show()

## 10. Hyperparameter sensitivity guide

| Parameter | Default | Effect |
|-----------|---------|--------|
| `n_d`, `n_a` | 16 | Larger → more capacity, slower. Decrease for small datasets. |
| `n_steps` | 4 | More steps = richer feature combinations. 3-5 is typical. |
| `gamma` | 1.5 | γ=1: features used once only. γ→∞: no prior penalty. 1.0-2.0 range. |
| `lambda_sparse` | 1e-3 | Higher → sparser masks (more interpretable, possibly less accurate). |
| `vbs` | 64 | Virtual batch size for Ghost BN. Should be < batch_size. |
| `lr` | 0.02 | Adam LR. Decay via `lr_decay` every `lr_decay_steps` global steps. |
| `patience` | 50 | Early stopping patience in epochs. Increase if training is noisy. |